# Prototype Run of Markov Model for Positive and Negative Datasets of Each Structural Class

### Set the Working Directory
The first cell verifies that the working directory is correctly set to the project root, Algorithms_Final_Project. Using the os package, it inspects the current working directory and resets it to the root if necessary. This step is essential because all subsequent file paths for data loading, model training, and results output are written relative to the project root. I added this check after encountering an immediate FileNotFoundError during the prototype run. Although my terminal was positioned at the correct root directory and all my filepaths were correctly written, notebook had been launched from within the src/ directory. As a result, every relative path silently resolved to the wrong location. This mismatch halted execution before any model logic could run and required extensive debugging to trace back to the notebook’s launch context.

Best practices emphasize running build, test, and execution commands from the project root so that the full directory structure is visible and consistently interpreted (Reitz, 2024). Because all of my relative paths were written with the root directory as the reference point, enforcing this directory check at the start of the notebook was the most reliable and non‑intrusive solution. This debugging step ultimately highlighted the importance of explicitly validating the working directory at the beginning of any execution workflow—especially in Jupyter, where the launch context may differ from the terminal environment.

In [ ]:
import os

cwd = os.getcwd()
# Ensure working dir is root, not src/ or notebooks/
if cwd.endswith("src") or cwd.endswith("notebooks"):
    os.chdir("..")

print("Working directory:", os.getcwd())


### Add src/ to Path
Setting the working directory to the project root fixes file paths, but it does not tell Python where to look for importable modules. Because all of my implementation code is organized within the src/ directory, I needed to add this directory to the Python path at runtime, for which I used the sys Python package.

In [ ]:
import sys

# Add src/ to Python path
sys.path.insert(0, "src")


### Import Submodules From Source Directory For Implementation
Now that the root is set to the working directory and src/ has been added to the Python path, I can import all modules needed to execute the prototype run. This includes data loading, model training, markov model, analyze data, terminal updates, and output to file. The notebook is now prepared for the prototype run.

In [ ]:
from data_loading import load_class_seqs, read_fasta
from train_models import train_all_models, train_markov
from analyze_data import (
    classify_test_set,
    compute_accuracy,
    compute_summary_stats,
    confusion_matrix,
    evaluate_model_performance
)
from output_to_file import write_all_outputs
from markov_model import (
    classify_sequence,
    log_likelihood,
    count_kmers,
    estimate_transition_probs
)



### Define Prototype FASTA Paths to Train and Test Model
In this cell, I define a dictionary containing the path to the Prototype Positive and Negative Promoter Class Datasets. In defining the FASTA path, I am initializing a dictionary in which the key denotes the class and label, while each value is a different sequence. Because each Promoter file has 300 total sequences, each of length 501bp, I am running a total prototype run of 600 sequences of uniform length.

 This dictionary acts as a routing map for the pipeline to know how to locate the files without having them hardcoded into the data loading module. This design choice allows for greater flexibility and modularity, as the data loading functions can simply reference this dictionary to access the correct files for each class and label, without needing to know the specific file paths. It also centralizes the file path definitions in one place, making it easier to update or modify the dataset locations in the future without having to change multiple parts of the codebase.

It is important to note that in the Prototype Run, the training data is the test data, so the same data is being run through train and the model. This is because this run is not for evaluating results, but rather, the pipeline, model interactions, model structure, output writing, classification loop, and directory structure.


In [ ]:
promoters = {
    "positive_promoters" : "data/prototype/promoters_positive.fa"
             }

control = {"negative_promoters" : "data/prototype/promoters_negative.fa"
            }

### Load Positive Promoter Prototype Sequences into Model

In [ ]:
training = load_class_seqs(promoters)


### Prepare Test Sequences and True Class Labels

In [ ]:
# Build test set and true label mapping for prototype evaluation

test_sequences = []  # Initialize list to hold test seqs
true_labels = {}  # Initialize dict mapping seq -> true class

# Extract every seq for every class
for class_label, seq_list in training.items():
    for rec in seq_list:
        seq_str = rec
        test_sequences.append(seq_str)
        true_labels[seq_str] = class_label

len(test_sequences), len(true_labels)


### Train Markov Models for All Classes and k Values

In [ ]:
# Train Markov model for positive promoter dataset over k = 1, 2, 3
# Parameter values
k_values = [1, 2, 3]  # k values under evaluation
alpha = 1  # Laplace smoothing constant

models = train_all_models(training, k_values, alpha)

models


### Classify Sequences

In [ ]:
# Classify all prototype sequences for each k value

results_k = []
for seq in test_sequences:
    pred, logL = classify_sequence(seq, models, k)
    results_k.append((seq, pred, logL))



In [ ]:
print(type(training["positive_promoters"][0]))


### Print Positive Promoter Results

In [ ]:
# Print positive results for each k value
for k in [1, 2, 3]:
    print(f"\n====*====> POSITIVE RESULTS FOR k = {k} <====*====")
    for seq, pred, logL in results_k[k]:
        print(f"k={k} | true=positive_promoters | pred={pred} | logL={logL}")


### Classify Control Data (Negative Promoters) for Comparison and Print

In [ ]:
control_results = []

for k in [1, 2, 3]:
    print(f"\n=== CONTROL RESULTS FOR k = {k} ===")
    for seq in control:
        pred, logL = classify_sequence(seq, models, k)
        control_results.append((k, seq, "control", pred, logL))
        print(f"k={k} | true=control | pred={pred} | logL={logL}")



### 9. Build Data Structure for Evaluation

In [ ]:
# Build full all_results structure required by evaluate_model_performance()

all_results = {}

for k in k_values:
    results_k = []
    raw = all_classification_results[k]

    for (seq, pred, score) in raw:
        # Placeholder class_scores: same score for all classes
        class_scores = {cls: score for cls in models[k].keys()}

        # Append full 4-tuple
        results_k.append((seq, pred, score, class_scores))

    all_results[k] = results_k

all_results



### 10a. Full Classification Analysis for a Single Order k

In [ ]:
k = 2

raw_classification_k = all_classification_results[k]

# 4-tuples for this k
results_k = all_results[k]

# Convert to dicts for per-k functions
results_dicts_k = [
    {
        "sequence": seq,
        "predicted_class": pred,
        "log_likelihood": score,
        "true_class": true_labels[seq]
    }
    for (seq, pred, score, _) in results_k
]

# Summary statistics (single k)
summary_stats_k = compute_summary_stats(results_dicts_k)

# Per-k accuracy
accuracy_k = compute_accuracy(results_dicts_k)

# Per-k confusion matrix
confusion_matrix_k = confusion_matrix(results_dicts_k)

#  Full cross-k metrics, restricted to single order k, including accuracy_vs_k, confusion_matrices, likelihood_distributions for k=2)
evaluation_k = evaluate_model_performance(
    {k: results_k},
    true_labels
)

summary_stats_k, accuracy_k, confusion_matrix_k, evaluation_k

### 10b. Compute Summary Statistics for All Orders k

In [ ]:
# Classifications per test seq for all orders k
raw_classification_all_k = all_classification_results

# Convert classify results for test seqs into dicts for all orders k
results_dicts_all_k = {
    k: [
        {
            "sequence": seq,
            "predicted_class": pred,
            "log_likelihood": score,
            "true_class": true_labels[seq]
        }
        for (seq, pred, score) in raw_classification_all_k[k]
    ]
    for k in k_values
}

# Summary statistics for all orders k
summary_stats_all_k = {
    k: compute_summary_stats(results_dicts_all_k[k])
    for k in k_values
}

# Accuracy for all orders k
accuracy_all_k = {
    k: compute_accuracy(results_dicts_all_k[k])
    for k in k_values
}

# Confusion matrices for all orders k
confusion_matrices_all_k = {
    k: confusion_matrix(results_dicts_all_k[k])
    for k in k_values
}

# Full cross-k evaluation metrics
# incl accuracy_vs_k, cross-k confusion matrices, likelihood distributions
evaluation_all_k = evaluate_model_performance(
    all_results,
    true_labels
)

# Return analysis results for all k
raw_classification_all_k, summary_stats_all_k, accuracy_all_k, confusion_matrices_all_k, evaluation_all_k


### 11. Write Results for Single k and All Orders k to Output Datafiles

In [ ]:
run = "prototype"        # this is the run name (prototype run)
base_dir = "results"

# write outputs for single k (k = 2)
run_name_single = "prototype_k2"

write_all_outputs(
    results=all_results[2],          # 4-tuples for k=2
    stats=summary_stats_k,           # summary stats for k=2
    metrics=evaluation_k,            # evaluation metrics for k=2
    dataset=run,                     # run folder name
    run_name=run_name_single,
    base_dir=base_dir
)

# write outputs for all k
run_name_all = "prototype_all_k"

# flatten all 4-tuples across k
all_results_flat = []
for k in k_values:
    all_results_flat.extend(all_results[k])

# build metrics dict in the structure write_all_outputs expects
combined_metrics_all_k = {
    "accuracy_vs_k": evaluation_all_k["accuracy_vs_k"],
    "confusion_matrices": evaluation_all_k["confusion_matrices"],
    "likelihood_distributions": evaluation_all_k["likelihood_distributions"]
}

# ---- FIX: DO NOT PASS summary_stats_all_k INTO write_all_outputs ----
# Instead, pass a dummy stats dict that satisfies write_summary_stats()
# but will be ignored because we skip summary stats for all-k.

dummy_stats = {
    "total": 0,
    "unclassified": 0,
    "neg_inf": 0,
    "class_counts": {}
}

write_all_outputs(
    results=all_results_flat,
    stats=dummy_stats,               # <-- FIXED
    metrics=combined_metrics_all_k,
    dataset=run,
    run_name=run_name_all,
    base_dir=base_dir
)

